# Candidate Embedding Generation

This notebook generates embeddings for 100k candidates using `BAAI/bge-small-en-v1.5`.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `candidates.jsonl` when prompted (Cell 3)

**Outputs** (auto-downloaded at the end):
- `embeddings.npy` — float32 matrix, shape (N, 384)
- `candidate_ids.json` — ordered list of candidate_ids matching row indices
- `text_blobs.json` — the text fed to the model (useful for debugging)

**Expected runtime on T4:** ~3 minutes for 100k candidates

## Cell 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS (Metal) GPU")
else:
    device = "cpu"
    print("No GPU found — running on CPU (will be slower)")

print(f"\nUsing device: {device}")

## Cell 2 — Install dependencies

In [ ]:
!pip install -q sentence-transformers tqdm

## Cell 3 — Mount Google Drive and set data path

**One-time setup (do this before running the cell):**
1. Go to [drive.google.com](https://drive.google.com)
2. Create a folder called `redrob` inside `My Drive`
3. Upload `candidates.jsonl` into it

**Then run this cell** — Colab will open a popup asking you to authorise Drive access. Do that once and it's mounted for the rest of the session.

In [ ]:
import os

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ── Update this if you saved the file to a different folder ──
    CANDIDATES_PATH = "/content/drive/MyDrive/redrob/candidates.jsonl"
else:
    CANDIDATES_PATH = "../data/raw/candidates.jsonl"

if not os.path.exists(CANDIDATES_PATH):
    raise FileNotFoundError(
        f"File not found: {CANDIDATES_PATH}\n"
        "Check that candidates.jsonl is in My Drive/redrob/ and Drive is mounted."
    )

print(f"Using: {CANDIDATES_PATH}")
print(f"File size: {os.path.getsize(CANDIDATES_PATH) / 1e6:.1f} MB")

## Cell 4 — Load candidates and build text blobs

In [ ]:
import json
from tqdm.auto import tqdm


def build_text_blob(c: dict) -> str:
    """Build a rich text representation of a candidate for embedding.
    
    Career descriptions carry the most signal for this JD — we weight them
    heavily. Skills section is included but we don't rely on it as the
    primary signal (the JD explicitly warns against keyword matching).
    """
    parts = []

    # --- Profile ---
    p = c.get("profile", {})
    if p.get("headline"):
        parts.append(p["headline"])
    if p.get("summary"):
        parts.append(p["summary"])

    # --- Career history (most important — free text descriptions) ---
    for role in c.get("career_history", []):
        title = role.get("title", "")
        company = role.get("company", "")
        industry = role.get("industry", "")
        company_size = role.get("company_size", "")
        desc = role.get("description", "")
        duration = role.get("duration_months", 0)
        years = round(duration / 12, 1)
        header = f"{title} at {company} ({company_size} employees, {industry}, {years} yrs)."
        parts.append(header)
        if desc:
            parts.append(desc)

    # --- Skills (with proficiency and duration for context) ---
    skills = c.get("skills", [])
    if skills:
        skill_strs = []
        for s in skills:
            name = s.get("name", "")
            prof = s.get("proficiency", "")
            dur = s.get("duration_months", 0)
            skill_strs.append(f"{name} ({prof}, {dur}mo)")
        parts.append("Skills: " + ", ".join(skill_strs))

    # --- Certifications ---
    certs = c.get("certifications", [])
    if certs:
        cert_strs = [f"{cert['name']} by {cert['issuer']} ({cert['year']})" for cert in certs]
        parts.append("Certifications: " + ", ".join(cert_strs))

    # --- Education ---
    for edu in c.get("education", []):
        inst = edu.get("institution", "")
        degree = edu.get("degree", "")
        field = edu.get("field_of_study", "")
        tier = edu.get("tier", "")
        parts.append(f"Education: {degree} in {field} from {inst} ({tier}).")

    return " ".join(parts)


print("Loading candidates...")
candidates = []
skipped = 0
with open(CANDIDATES_PATH) as f:
    for line_num, line in enumerate(tqdm(f, desc="Reading"), 1):
        line = line.strip()
        if not line:
            continue
        try:
            candidates.append(json.loads(line))
        except json.JSONDecodeError:
            skipped += 1

print(f"\nLoaded:  {len(candidates):,} candidates")
if skipped:
    print(f"Skipped: {skipped} malformed lines (truncated during upload — re-upload if > 5)")

print("Building text blobs...")
candidate_ids = [c["candidate_id"] for c in candidates]
text_blobs = [build_text_blob(c) for c in tqdm(candidates, desc="Building blobs")]

# Sanity check
sample_idx = 0
print(f"\n--- Sample blob for {candidate_ids[sample_idx]} ---")
print(text_blobs[sample_idx][:800])
print("...")
avg_len = sum(len(t) for t in text_blobs) / len(text_blobs)
print(f"\nAvg blob length: {avg_len:.0f} chars")

## Cell 5 — Load model and generate embeddings

This is the main step. Expected time on T4 GPU: ~3 minutes.

In [ ]:
import gc

# Free the raw candidates list — text_blobs has everything we need
del candidates
gc.collect()
print("Freed candidates from RAM")

In [ ]:
import numpy as np
import time
import gc
import os
from sentence_transformers import SentenceTransformer

# Swap this one line when ready to upgrade:
# "all-MiniLM-L6-v2"      — 22MB,  384 dims, ~3 min  (current)
# "BAAI/bge-large-en-v1.5" — 1.3GB, 1024 dims, ~15 min (upgrade)
MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 256
CHUNK_SIZE = 25_000

print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")
print(f"Batch size: {BATCH_SIZE}, Chunk size: {CHUNK_SIZE}")
print(f"Total candidates: {len(text_blobs):,}")
print()

os.makedirs("chunks", exist_ok=True)
start = time.time()

chunk_starts = list(range(0, len(text_blobs), CHUNK_SIZE))
for i, chunk_start in enumerate(chunk_starts):
    chunk_end = min(chunk_start + CHUNK_SIZE, len(text_blobs))
    chunk_path = f"chunks/embeddings_chunk_{i}.npy"

    if os.path.exists(chunk_path):
        print(f"Chunk {i} already exists — skipping")
        continue

    print(f"Encoding chunk {i+1}/{len(chunk_starts)}: {chunk_start:,}–{chunk_end:,}")
    chunk_emb = model.encode(
        text_blobs[chunk_start:chunk_end],
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    np.save(chunk_path, chunk_emb)
    print(f"  Saved — shape {chunk_emb.shape}, {chunk_emb.nbytes/1e6:.0f} MB")
    del chunk_emb
    gc.collect()

print("\nMerging chunks...")
embeddings = np.concatenate([
    np.load(f"chunks/embeddings_chunk_{i}.npy") for i in range(len(chunk_starts))
])

elapsed = time.time() - start
print(f"Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Embeddings shape: {embeddings.shape}")
print(f"Memory: {embeddings.nbytes / 1e6:.1f} MB")

## Cell 6 — Validate embeddings

In [ ]:
JD_QUERY = (
    "Represent this sentence for searching relevant passages: "
    "Senior AI Engineer with production experience in embeddings-based retrieval, "
    "vector databases, hybrid search, LLM reranking, and evaluation frameworks. "
    "5-9 years experience at product companies (not consulting). "
    "Has shipped ranking or recommendation systems to real users."
)

query_embedding = model.encode(
    JD_QUERY,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

scores = embeddings @ query_embedding
top_indices = scores.argsort()[::-1][:10]

print("Top 10 candidates for the JD query:\n")
for rank, idx in enumerate(top_indices, 1):
    cid = candidate_ids[idx]
    score = scores[idx]
    blob_preview = text_blobs[idx][:120].replace("\n", " ")
    print(f"{rank:2}. {cid}  score={score:.4f}  {blob_preview}...")

print("\nTop results should look like AI/ML engineers at product companies — not consultants.")

## Cell 7 — Save outputs

In [ ]:
import json

# Save embeddings matrix
np.save("embeddings.npy", embeddings)
print(f"Saved embeddings.npy — {embeddings.shape}, {embeddings.nbytes/1e6:.1f} MB")

# Save ordered candidate IDs (row i in embeddings.npy = candidate_ids[i])
with open("candidate_ids.json", "w") as f:
    json.dump(candidate_ids, f)
print(f"Saved candidate_ids.json — {len(candidate_ids):,} entries")

# Save text blobs (useful for BM25 and debugging — not needed for pgvector)
with open("text_blobs.json", "w") as f:
    json.dump(text_blobs, f)
print(f"Saved text_blobs.json — {len(text_blobs):,} entries")

# Save metadata
metadata = {
    "model": MODEL_NAME,
    "embedding_dim": int(embeddings.shape[1]),
    "num_candidates": int(embeddings.shape[0]),
    "normalized": True,
    "device_used": device,
    "elapsed_seconds": round(elapsed, 1),
}
with open("embeddings_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved embeddings_metadata.json")
print(f"\nMetadata: {json.dumps(metadata, indent=2)}")

## Cell 8 — Download to your local machine

This will trigger browser downloads for all 4 files.

**Save them to:** `candidate-ranking/data/processed/`

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Downloading files...")
    files.download("embeddings.npy")           # ~150 MB
    files.download("candidate_ids.json")       # ~2 MB
    files.download("text_blobs.json")          # ~200 MB
    files.download("embeddings_metadata.json") # tiny
    print("\nAll files downloaded.")
    print("Move them to: candidate-ranking/data/processed/")
else:
    import shutil, os
    os.makedirs("../data/processed", exist_ok=True)
    for fname in ["embeddings.npy", "candidate_ids.json", "text_blobs.json", "embeddings_metadata.json"]:
        shutil.move(fname, f"../data/processed/{fname}")
    print("Files moved to data/processed/")